## Matplotlib M+L Flaws Robustness Study

In [ ]:
# === Robustness scoring (images, dual summaries & similarity; streaming/resumable) ===

import sys
import csv as _pycsv
try:
    _pycsv.field_size_limit(sys.maxsize)
except OverflowError:
    _pycsv.field_size_limit(2**31 - 1)

import os, ast, re, sys, time, hashlib, math, io, traceback
import pandas as pd

# Progress bar (optional)
try:
    from tqdm import tqdm
except Exception:
    tqdm = None  # fallback to plain prints if tqdm isn't installed

# ---------- CONFIG ----------
SRC = 'github_matplotlib_audit_1'  # base name or .csv
LLM_MIN_INTERVAL = float(os.getenv("LLM_MIN_INTERVAL", "6.0"))
MODEL_NAME = os.getenv("GEMINI_MODEL", "gemini-1.5-flash")
MAX_ROWS = int(os.getenv("MAX_ROWS", "0"))         # 0 = all; else stop early
PRINT_EVERY = int(os.getenv("PRINT_EVERY", "50"))  # progress cadence
SAMPLE_N = int(os.getenv("SAMPLE_N", "10"))        # sample size; 0 = all
SAMPLE_SEED = int(os.getenv("SAMPLE_SEED", "1337"))
GRAPH_TIMEOUT = float(os.getenv("GRAPH_TIMEOUT", "10"))  # per-row exec timeout (sec)
GRAPHS_DIR = os.getenv("GRAPHS_DIR", "./graphs")

_LLM_LAST = 0.0  # do not touch

os.makedirs(GRAPHS_DIR, exist_ok=True)

# ---------- Gemini setup (guarded) ----------
HAVE_GEMINI = False
try:
    import google.generativeai as genai
    api_key = os.getenv("GOOGLE_API_KEY", None)
    if not api_key:
        from GEMINI_API_KEY import GEMINI_API_KEY as api_key  # optional fallback
    if not api_key:
        raise RuntimeError("No Gemini API key found in env or GEMINI_API_KEY.py")
    genai.configure(api_key=api_key)
    client = genai.GenerativeModel(MODEL_NAME)
    HAVE_GEMINI = True
except Exception as e:
    print(f"⚠️ Gemini disabled: {e}")
    HAVE_GEMINI = False

# ---- Percent similarity helper (0..100). Tries API first, then local model. ----
_SIM_API_URL = os.getenv("SIM_API_URL", "http://127.0.0.1:8002/compute_semantic_similarity")

def _cosine_sim_pct(sentence1: str, sentence2: str):
    # 1) Try your FastAPI service (returns raw cosine)
    if _SIM_API_URL:
        try:
            import requests
            r = requests.get(_SIM_API_URL,
                             params={"sentence1": sentence1, "sentence2": sentence2},
                             timeout=10)
            if r.ok:
                cos = float(r.json())
                cos = max(-1.0, min(1.0, cos))           # clamp
                pct = ((cos + 1.0) / 2.0) * 100.0        # map [-1,1] -> [0,100]
                return round(pct, 2)
        except Exception:
            pass

# ---------- Rules ----------
NONCONTEXTUAL_CODES = [
    "MISSING_TITLE","MISSING_XLABEL","MISSING_YLABEL","MISSING_LEGEND",
    "FONTSIZE_TOO_SMALL","FIGSIZE_TOO_SMALL","INSUFFICIENT_COLOR_CONTRAST",
    "ANIMATIONS","INVERTED_Y_AXIS","TRUNCATED_Y_AXIS","3D_EFFECTS",
    "TAMPERED_ASPECT_RATIO","DUAL_Y_AXES",
]
CONTEXTUAL_CODES = [
    "BIASED_TITLE","MISLEADING_ANNOTATIONS","DECEPTIVE_LABELS","FRAMING_BIAS",
    "INVERTED_AXES","TRUNCATED_AXES","ASPECT_RATIO_DISTORTION","DUAL_AXES",
    "NON_SEQUENTIAL_AXIS",
]
ALL_RULE_COLS = NONCONTEXTUAL_CODES + CONTEXTUAL_CODES

RULE_DESCRIPTIONS = {
    # Non-contextual
    "MISSING_TITLE": "Title is missing or empty.",
    "MISSING_XLABEL": "X-axis label is missing or unhelpful.",
    "MISSING_YLABEL": "Y-axis label is missing or unhelpful.",
    "MISSING_LEGEND": "Legend missing despite multiple series.",
    "FONTSIZE_TOO_SMALL": "Text uses too small font sizes (readability).",
    "FIGSIZE_TOO_SMALL": "Figure size too small for readability.",
    "INSUFFICIENT_COLOR_CONTRAST": "Color choices may be low-contrast.",
    "ANIMATIONS": "Contains animations which may harm accessibility.",
    "INVERTED_Y_AXIS": "Y-axis inverted which can flip meaning.",
    "TRUNCATED_Y_AXIS": "Y-axis does not start at zero (may exaggerate).",
    "3D_EFFECTS": "3D effects may distort perception.",
    "TAMPERED_ASPECT_RATIO": "Aspect ratio altered to exaggerate trends.",
    "DUAL_Y_AXES": "Dual y-axes may suggest false correlation.",
    # Contextual
    "BIASED_TITLE": "Emotionally slanted or biased title.",
    "MISLEADING_ANNOTATIONS": "Annotations imply unjustified causality/relations.",
    "DECEPTIVE_LABELS": "Ambiguous or misleading axis labels/categories.",
    "FRAMING_BIAS": "External text framing introduces bias.",
    "INVERTED_AXES": "Axis inversion that misleads interpretation.",
    "TRUNCATED_AXES": "Axis baseline not zero (exaggeration).",
    "ASPECT_RATIO_DISTORTION": "Aspect ratio distorts slope/shape perception.",
    "DUAL_AXES": "Two different y-axes suggest correlation.",
    "NON_SEQUENTIAL_AXIS": "Axis categories are out of logical order.",
}

# ---------- Helpers ----------
def _maybe_wait_for_llm():
    global _LLM_LAST
    elapsed = time.time() - _LLM_LAST
    if elapsed < LLM_MIN_INTERVAL:
        time.sleep(LLM_MIN_INTERVAL - elapsed)

def _sanitize_one_line(s: str) -> str:
    return re.sub(r'\s+', ' ', (s or '')).strip()

def _pick_masked_rule(row_id: int, detected_rules):
    if not detected_rules:
        return ""
    arr = sorted(set(detected_rules))
    h = int(hashlib.md5(str(row_id).encode('utf-8')).hexdigest(), 16)
    return arr[h % len(arr)]

# ---------- Code scanners (same as before) ----------
def find_noncontextual_flaws(mpl_file: str):
    flaws = []
    mpl_index = mpl_file.find("import matplotlib")
    if mpl_index == -1:
        mpl_index = mpl_file.find("from matplotlib import")
    if mpl_index == -1:
        return flaws
    scan_text = mpl_file[mpl_index:]

    # Descriptive labels
    for fn in ["title", "xlabel", "ylabel"]:
        match = re.search(rf"{fn}\s*\(\s*['\"]([^'\"]*)['\"]", scan_text)
        if match:
            txt = match.group(1).strip().lower()
            if not txt or txt in ["x", "y", "series 1"]:
                flaws.append(f"MISSING_{fn.upper()}")
        else:
            flaws.append(f"MISSING_{fn.upper()}")

    # Legend required when multiple series
    plot_count = len(re.findall(r'plot\s*\(', scan_text))
    scatter_count = len(re.findall(r'scatter\s*\(', scan_text))
    if (plot_count + scatter_count > 1) and 'legend(' not in scan_text:
        flaws.append("MISSING_LEGEND")

    # Font size
    font_matches = re.findall(r'fontsize\s*=\s*(\d+)', scan_text)
    if any(int(size) < 15 for size in font_matches):
        flaws.append("FONTSIZE_TOO_SMALL")

    # Figure size
    fig_match = re.search(r'figsize\s*=\s*\(\s*([\d.]+)\s*,\s*([\d.]+)\s*\)', scan_text)
    if fig_match:
        w, h = float(fig_match.group(1)), float(fig_match.group(2))
        if w < 8 or h < 5:
            flaws.append("FIGSIZE_TOO_SMALL")

    # High-contrast colors
    color_matches = re.findall(r'color\s*=\s*[\'"]([^\'"]+)[\'"]', scan_text)
    safe_colors = {"#000000", "#0072B2", "#009E73", "#D55E00", "black", "blue", "green", "orange"}
    if any(color.lower() not in safe_colors for color in color_matches):
        flaws.append("INSUFFICIENT_COLOR_CONTRAST")

    # No animations
    if "FuncAnimation" in scan_text or "animation." in scan_text:
        flaws.append("ANIMATIONS")

    # Inverted Y-axis
    if re.search(r'\.\s*invert_yaxis\s*\(', scan_text):
        flaws.append("INVERTED_Y_AXIS")

    # Truncated Y-axis
    for m in re.finditer(r'(?:set_)?ylim\s*\(\s*([\-]?\d+(?:\.\d+)?)\s*,', scan_text):
        lower = float(m.group(1))
        if abs(lower) > 1e-6:
            flaws.append("TRUNCATED_Y_AXIS")
            break

    # 3D effects
    if (re.search(r'["\']\s*3d\s*["\']', scan_text) or "Axes3D" in scan_text or "plot_surface(" in scan_text):
        flaws.append("3D_EFFECTS")

    # Aspect ratio
    fig_aspect = re.search(r'figsize\s*=\s*\(\s*([\d.]+)\s*,\s*([\d.]+)\s*\)', scan_text)
    if fig_aspect:
        w, h = float(fig_aspect.group(1)), float(fig_aspect.group(2))
        if h != 0:
            ratio = w / h
            if ratio < 0.5 or ratio > 2.0:
                flaws.append("TAMPERED_ASPECT_RATIO")
    if re.search(r'set_aspect\s*\(|aspect\s*=', scan_text):
        flaws.append("TAMPERED_ASPECT_RATIO")

    # Dual Y
    if re.search(r'twin[xy]\s*\(', scan_text) or 'secondary_y=True' in scan_text:
        flaws.append("DUAL_Y_AXES")

    return list(sorted(set(flaws)))

def find_contextual_flaws(mpl_file: str):
    mpl_index = mpl_file.find("import matplotlib")
    if mpl_index == -1:
        mpl_index = mpl_file.find("from matplotlib import")
    if mpl_index == -1 or not HAVE_GEMINI:
        return []
    scan_text = mpl_file[mpl_index:]
    _maybe_wait_for_llm()
    template = """You are an expert in data visualization integrity. Output only a Python list of RULE_CODEs violated by the following Matplotlib code; return NONE if none.
Valid RULE_CODEs: BIASED_TITLE,MISLEADING_ANNOTATIONS,DECEPTIVE_LABELS,FRAMING_BIAS,INVERTED_AXES,TRUNCATED_AXES,ASPECT_RATIO_DISTORTION,DUAL_AXES,NON_SEQUENTIAL_AXIS.
Code:
{code}"""
    prompt = template.format(code=scan_text)
    try:
        raw = client.generate_content(prompt).text or ""
    except Exception as e:
        msg = str(e)
        if ("ResourceExhausted" in msg) or ("429" in msg) or ("quota" in msg.lower()) or ("rate" in msg.lower()) or ("Timeout" in msg) or ("timed out" in msg.lower()):
            time.sleep(LLM_MIN_INTERVAL * 1.5)
            return []
        return []
    finally:
        global _LLM_LAST
        _LLM_LAST = time.time()

    if raw.strip().upper() == "NONE":
        return []
    try:
        parsed = ast.literal_eval(raw)
        if isinstance(parsed, list):
            return list(sorted(set(str(x).strip().upper() for x in parsed)))
    except Exception:
        return []
    return []

# ---------- Rendering (subprocess) ----------
from multiprocessing import Process, Queue
import matplotlib
matplotlib.use("Agg")  # headless
import matplotlib.pyplot as plt

def _exec_and_save(code: str, out_prefix: str, queue: Queue):
    errs = None
    saved = []
    try:
        ns = {}
        try:
            exec(code, ns)
        except Exception as e:
            errs = f"exec-error: {e}"

        figs = [plt.figure(n) for n in plt.get_fignums()]
        for i, fig in enumerate(figs, start=1):
            path = os.path.join(GRAPHS_DIR, f"{out_prefix}_{i}.png")
            try:
                fig.savefig(path)
                saved.append(path)
            except Exception as e:
                errs = (errs or "") + f" | save-error-{i}: {e}"
        plt.close('all')
    finally:
        queue.put((saved, errs))

def render_images_from_code(code: str, out_prefix: str, timeout: float = GRAPH_TIMEOUT):
    q = Queue()
    p = Process(target=_exec_and_save, args=(code, out_prefix, q))
    p.start()
    p.join(timeout=timeout)
    if p.is_alive():
        p.terminate(); p.join()
        return [], "timeout"
    if not q.empty():
        imgs, err = q.get()
        return imgs, err
    return [], "unknown"

# ---------- Masking (auto “fix/hide” for some non-contextual rules) ----------
def patch_code_for_mask(original_code: str, rule: str) -> str:
    # We don't try to surgically edit the original text; instead we append a post-pass
    # that walks the current figure/axes and normalizes elements.
    post = ["import matplotlib.pyplot as _plt_",
            "_figs_ = [_plt_.figure(n) for n in _plt_.get_fignums()]",
            "import numpy as _np_  # might be unused; safe",
            ""]

    if rule == "MISSING_TITLE":
        post += ["[ax.set_title('') for f in _figs_ for ax in f.get_axes()]"]
    elif rule == "MISSING_XLABEL":
        post += ["[ax.set_xlabel('') for f in _figs_ for ax in f.get_axes()]"]
    elif rule == "MISSING_YLABEL":
        post += ["[ax.set_ylabel('') for f in _figs_ for ax in f.get_axes()]"]
    elif rule == "MISSING_LEGEND":
        post += [
            "for f in _figs_:",
            "  for ax in f.get_axes():",
            "    handles, labels = ax.get_legend_handles_labels()",
            "    if not labels:",
            "      lines = ax.get_lines()",
            "      if lines:",
            "        handles = lines",
            "        labels = [f'Series {i+1}' for i in range(len(lines))]",
            "    if handles and labels:",
            "      ax.legend(handles, labels)",
        ]
    elif rule == "FONTSIZE_TOO_SMALL":
        post += [
            "for f in _figs_:",
            "  for ax in f.get_axes():",
            "    ax.title.set_fontsize(16)",
            "    ax.xaxis.label.set_fontsize(14)",
            "    ax.yaxis.label.set_fontsize(14)",
            "    for tick in ax.get_xticklabels()+ax.get_yticklabels():",
            "      tick.set_fontsize(12)",
        ]
    elif rule == "FIGSIZE_TOO_SMALL":
        post += [
            "for f in _figs_:",
            "  try: f.set_size_inches(8,5, forward=True)",
            "  except Exception: pass",
        ]
    elif rule == "INSUFFICIENT_COLOR_CONTRAST":
        post += [
            "for f in _figs_:",
            "  for ax in f.get_axes():",
            "    for ln in ax.get_lines(): ln.set_color('black')",
            "    for coll in ax.collections: ",
            "      try: coll.set_color('black')",
            "      except Exception: pass",
        ]
    elif rule == "INVERTED_Y_AXIS":
        post += [
            "for f in _figs_:",
            "  for ax in f.get_axes():",
            "    lo, hi = ax.get_ylim()",
            "    if lo > hi: ax.set_ylim(min(lo,hi), max(lo,hi))",
        ]
    elif rule == "TRUNCATED_Y_AXIS":
        post += [
            "for f in _figs_:",
            "  for ax in f.get_axes():",
            "    lo, hi = ax.get_ylim()",
            "    ax.set_ylim(bottom=min(0, lo, hi))",
        ]
    elif rule == "TAMPERED_ASPECT_RATIO":
        post += [
            "for f in _figs_:",
            "  for ax in f.get_axes():",
            "    try: ax.set_aspect('auto')",
            "    except Exception: pass",
            "  try: f.set_size_inches(8,5, forward=True)",
            "  except Exception: pass",
        ]
    elif rule == "DUAL_Y_AXES":
        # Not reliably fixable post-hoc; do nothing.
        post += []
    elif rule in CONTEXTUAL_CODES:
        # We don't patch code for contextual rules.
        post += []
    else:
        post += []

    return original_code + "\n\n# --- auto-mask post-pass ---\n" + "\n".join(post) + "\n"

# ---------- Summarization on IMAGES ----------
_SUMMARY_PROMPT = """You are a screen reader and came across this data visualization. Describe it in 1-2 sentences using simple, friendly language. Mention what kind of visualization it is, its title (if any), any highs and lows, and what the overall pattern seems to be. Start with "A [visualization type] shows…" or "A [visualization type] titled [title] shows…". If it is not a data visualization, say "N/A"."""

def _withhold_clause(rule_code: str):
    if not rule_code:
        return ""
    desc = RULE_DESCRIPTIONS.get(rule_code, rule_code)
    return f"\nCRITICAL: Do NOT mention or allude to this violation: {rule_code}: {desc}.\n"

def summarize_from_image(image_path: str, withhold_rule: str = "") -> str:
    if not HAVE_GEMINI or not os.path.exists(image_path):
        return ""
    _maybe_wait_for_llm()
    try:
        uploaded = genai.upload_file(image_path)
        prompt = _SUMMARY_PROMPT + (_withhold_clause(withhold_rule) if withhold_rule else "")
        resp = client.generate_content([prompt, uploaded])
        text = getattr(resp, "text", "") or ""
        return _sanitize_one_line(text)
    except Exception as e:
        msg = str(e)
        if ("ResourceExhausted" in msg) or ("429" in msg) or ("quota" in msg.lower()) or ("rate" in msg.lower()) or ("Timeout" in msg) or ("timed out" in msg.lower()):
            time.sleep(LLM_MIN_INTERVAL * 1.5)
            return ""
        return ""
    finally:
        global _LLM_LAST
        _LLM_LAST = time.time()

# ---------- Load input ----------
fname = SRC if os.path.exists(SRC) else (SRC + '.csv' if os.path.exists(SRC + '.csv') else SRC)
try:
    df = pd.read_csv(fname)
except Exception:
    df = pd.read_csv(SRC + '.csv')
    fname = SRC + '.csv'

print(f"Loaded: {fname}  (rows={len(df)})")

# Detect code column
POSSIBLE_CODE_COLS = ["matplotlib_code","code","file_content","content","snippet","source","body","text"]
code_col = None
for c in POSSIBLE_CODE_COLS:
    if c in df.columns:
        code_col = c; break
if code_col is None:
    for c in df.columns:
        if df[c].dtype == object:
            sample = df[c].dropna().astype(str).head(10)
            if sample.str.contains("import matplotlib|from matplotlib", regex=True, case=False).any():
                code_col = c; break
if code_col is None:
    raise ValueError("Could not find a column with Matplotlib code. Add a column named e.g., 'matplotlib_code'.")

print(f"Using code column: {code_col}")

# Sampling
if SAMPLE_N and SAMPLE_N > 0 and SAMPLE_N < len(df):
    df = df.sample(n=SAMPLE_N, random_state=SAMPLE_SEED)
    print(f"Sampling: taking {len(df)} rows (SAMPLE_N={SAMPLE_N}, seed={SAMPLE_SEED})")

# ---------- Prepare streaming output ----------
base, ext = os.path.splitext(fname)
out = f"{base}_scored_stream{ext or '.csv'}"

orig_cols = list(df.columns)
new_cols = [
    "FULL_SUMMARY","MASKED_SUMMARY","MASKED_RULE","SUMMARY_SIMILARITY","SUMMARY_PAIR_ORDER",
    "FULL_IMAGE","MASKED_IMAGE"
]
header_cols = (["row_id"] + orig_cols +
               [c for c in ALL_RULE_COLS if c not in orig_cols] +
               ["NONCONTEXTUAL_LIST","CONTEXTUAL_LIST"] +
               new_cols)

processed_ids = set()
if os.path.exists(out):
    try:
        prev = pd.read_csv(out, usecols=["row_id"])
        processed_ids = set(prev["row_id"].dropna().astype(int).tolist())
        print(f"Resuming; already have {len(processed_ids)} rows in {out}")
    except Exception as e:
        print(f"Note: couldn't read existing {out} for resume ({e}); starting fresh.")

if not os.path.exists(out):
    pd.DataFrame(columns=header_cols).to_csv(out, index=False)

# ---------- Main ----------


def score_row(code: str):
    if not isinstance(code, str):
        return [], []
    nonctx = find_noncontextual_flaws(code)
    ctx = find_contextual_flaws(code) if HAVE_GEMINI else []
    return nonctx, ctx

rows_done = 0
for idx, row in df.iterrows():
    row_id = int(idx)
    if row_id in processed_ids:
        continue

    code_raw = row[code_col] if isinstance(row[code_col], str) else ""
    # focus on matplotlib portion if present
    mpl_index = code_raw.find("import matplotlib")
    if mpl_index == -1:
        mpl_index = code_raw.find("from matplotlib import")
    code = code_raw[mpl_index:] if mpl_index != -1 else code_raw

    nonctx_list, ctx_list = score_row(code)

    # Build base output
    out_row = dict(row)
    out_row["row_id"] = row_id
    for r in NONCONTEXTUAL_CODES:
        out_row[r] = 1 if r in nonctx_list else 0
    for r in CONTEXTUAL_CODES:
        out_row[r] = 1 if r in ctx_list else 0
    out_row["NONCONTEXTUAL_LIST"] = ";".join(nonctx_list)
    out_row["CONTEXTUAL_LIST"] = ";".join(ctx_list)

    full_img = ""
    masked_img = ""
    full_summary = ""
    masked_summary = ""
    masked_rule = ""
    pair_order = "sentence1=FULL, sentence2=MASKED"

    # ---- Render FULL image
    full_prefix = f"row{row_id}_full"
    imgs, err = render_images_from_code(code, full_prefix, timeout=GRAPH_TIMEOUT)
    if imgs:
        full_img = imgs[0]  # take first figure
    # ---- Choose rule to mask (from union)
    detected_all = sorted(set(nonctx_list + ctx_list))
    masked_rule = _pick_masked_rule(row_id, detected_all)

    # ---- Render MASKED image (patch only for non-contextual we can affect)
    masked_prefix = f"row{row_id}_masked_{masked_rule or 'none'}"
    if masked_rule in NONCONTEXTUAL_CODES:
        patched = patch_code_for_mask(code, masked_rule)
        m_imgs, m_err = render_images_from_code(patched, masked_prefix, timeout=GRAPH_TIMEOUT)
        if m_imgs:
            masked_img = m_imgs[0]
    else:
        # contextual or none: reuse original code image (no code patch)
        masked_img = full_img

    # ---- Summaries from IMAGES
    if full_img:
        full_summary = summarize_from_image(full_img, withhold_rule="") if HAVE_GEMINI else ""
    if masked_img:
        # For contextual rules, we don't patch code; still apply a withhold instruction so the flaw is "missing" in text.
        withhold = masked_rule if masked_rule in CONTEXTUAL_CODES else ""
        masked_summary = summarize_from_image(masked_img, withhold_rule=withhold) if HAVE_GEMINI else ""

    # ---- Similarity (S1 = full, S2 = masked)
    sim_val = _cosine_sim_pct(full_summary, masked_summary) if (full_summary and masked_summary) else ""


    # ---- Fill new cols
    out_row["FULL_SUMMARY"] = full_summary
    out_row["MASKED_SUMMARY"] = masked_summary
    out_row["MASKED_RULE"] = masked_rule
    out_row["SUMMARY_SIMILARITY"] = sim_val
    out_row["SUMMARY_PAIR_ORDER"] = pair_order
    out_row["FULL_IMAGE"] = full_img
    out_row["MASKED_IMAGE"] = masked_img

    # ---- Append
    pd.DataFrame([out_row])[header_cols].to_csv(out, mode="a", header=False, index=False)

    rows_done += 1
    if rows_done % PRINT_EVERY == 0:
        print(f"Processed {rows_done} rows (last row_id={row_id}) → {out}")

    if MAX_ROWS and rows_done >= MAX_ROWS:
        print(f"Hit MAX_ROWS={MAX_ROWS}; stopping early.")
        break




print(f"✅ Done. Appended {rows_done} new rows → {out}")


Loaded: github_matplotlib_audit_1.csv  (rows=14956)
Using code column: code
Sampling: taking 10 rows (SAMPLE_N=10, seed=1337)
Resuming; already have 1 rows in github_matplotlib_audit_1_scored_stream.csv


In [9]:
import pandas as pd
df1 = pd.read_csv("github_matplotlib_audit_1_scored_stream.csv",
                  engine="python", on_bad_lines="warn")  # or "skip"


C:\Users\Administrator\AppData\Local\Temp\ipykernel_4552\871008538.py:2: ParserWarning: Skipping line 12: Expected 33 fields in line 12, saw 34

  df1 = pd.read_csv("github_matplotlib_audit_1_scored_stream.csv",
C:\Users\Administrator\AppData\Local\Temp\ipykernel_4552\871008538.py:2: ParserWarning: Skipping line 13: Expected 33 fields in line 13, saw 34

  df1 = pd.read_csv("github_matplotlib_audit_1_scored_stream.csv",
C:\Users\Administrator\AppData\Local\Temp\ipykernel_4552\871008538.py:2: ParserWarning: Skipping line 14: Expected 33 fields in line 14, saw 34

  df1 = pd.read_csv("github_matplotlib_audit_1_scored_stream.csv",
C:\Users\Administrator\AppData\Local\Temp\ipykernel_4552\871008538.py:2: ParserWarning: Skipping line 15: Expected 33 fields in line 15, saw 34

  df1 = pd.read_csv("github_matplotlib_audit_1_scored_stream.csv",
C:\Users\Administrator\AppData\Local\Temp\ipykernel_4552\871008538.py:2: ParserWarning: Skipping line 16: Expected 33 fields in line 16, saw 34

  df1 =

In [10]:
df1.head()

,row_id,repo,filename,pushed_date,code,MISSING_TITLE,MISSING_XLABEL,MISSING_YLABEL,MISSING_LEGEND,FONTSIZE_TOO_SMALL,...,TRUNCATED_AXES,ASPECT_RATIO_DISTORTION,DUAL_AXES,NON_SEQUENTIAL_AXIS,NONCONTEXTUAL_LIST,CONTEXTUAL_LIST,SUMMARY_FULL,MASKED_RULE,SUMMARY_MASKED,SUMMARY_SIMILARITY
0,3921,safdaraliniazi/Blue-Bank-Loan-Analysis,bank.py,2023-02-10,"# -*- coding: utf-8 -*-\r\n""""""\r\nCreated on W...",1,1,1,0,0,...,0,0,0,0,MISSING_TITLE;MISSING_XLABEL;MISSING_YLABEL,NaN,A bar chart shows the number of people in each...,MISSING_YLABEL,A bar chart shows the number of people in each...,NaN
1,14845,krottapalli1923G/IMDB-Movie-Ratings-Analysis,step5_genre_ratings.py,2025-01-30,import pandas as pd\nimport numpy as np\nimpor...,0,0,0,0,0,...,0,0,0,0,NaN,NaN,A bar chart titled “🎭 Average Rating Per Genre...,NaN,A bar chart titled “🎭 Average Rating Per Genre...,NaN
2,6669,Izzenn/Graphs,PYPT graphs.py,2023-09-24,import matplotlib.pyplot as plt\nimport numpy ...,0,1,1,0,0,...,1,0,0,0,INSUFFICIENT_COLOR_CONTRAST;MISSING_XLABEL;MIS...,TRUNCATED_AXES,NaN,MISSING_XLABEL,A line graph shows two lines with error bars. ...,NaN
3,10315,VSingh1012/DataFy---Data-Research-Project,BarFile.py,2024-06-22,import tkinter as tk\nfrom tkinter import file...,0,1,1,0,0,...,1,0,0,0,INSUFFICIENT_COLOR_CONTRAST;MISSING_XLABEL;MIS...,TRUNCATED_AXES,A bar graph shows the relationship between two...,TRUNCATED_AXES,A bar graph shows the relationship between two...,NaN
4,14872,ulyssetresor13/Medical-Data-Visualizer,medical_data_visualizer.py,2025-01-31,import pandas as pd\nimport seaborn as sns\nim...,1,1,1,1,0,...,0,0,0,0,MISSING_LEGEND;MISSING_TITLE;MISSING_XLABEL;MI...,NaN,NaN,MISSING_LEGEND,A bar chart shows the total counts of differen...,NaN


In [ ]:
df2 = pd.read_csv("github_matplotlib_audit_1_scored_stream.csv")
df2.head()

In [ ]:
# === Robustness scoring (streaming/resumable) — with masked summaries ===
# Appends one row at a time to output CSV so you keep progress even if it crashes.

import os, ast, re, sys, time, hashlib, random
import pandas as pd

# ---------- CONFIG ----------
SRC = 'github_matplotlib_audit_1'  # base name or .csv
LLM_MIN_INTERVAL = float(os.getenv("LLM_MIN_INTERVAL", "6.0"))  # throttle between LLM calls (sec)
MODEL_NAME = os.getenv("GEMINI_MODEL", "gemini-1.5-flash")
MAX_ROWS = int(os.getenv("MAX_ROWS", "0"))  # 0 = all rows; else stop early for testing
PRINT_EVERY = int(os.getenv("PRINT_EVERY", "50"))  # progress print cadence
SAMPLE_N = int(os.getenv("SAMPLE_N", "10"))  # default: grab a small random sample of 10

_LLM_LAST = 0.0  # do not touch

# ---------- Gemini setup (guarded) ----------
HAVE_GEMINI = False
try:
    import google.generativeai as genai
    api_key = os.getenv("GOOGLE_API_KEY", None)
    if not api_key:
        from GEMINI_API_KEY import GEMINI_API_KEY as api_key  # optional fallback
    if not api_key:
        raise RuntimeError("No Gemini API key found in env or GEMINI_API_KEY.py")
    genai.configure(api_key=api_key)
    client = genai.GenerativeModel(MODEL_NAME)
    HAVE_GEMINI = True
except Exception as e:
    print(f"⚠️ Gemini disabled: {e}")
    HAVE_GEMINI = False

# ---------- Rule dictionaries ----------
# Canonicalize similar codes so summaries use a consistent vocabulary.
_CANON_MAP = {
    "INVERTED_Y_AXIS": "INVERTED_AXES",
    "TRUNCATED_Y_AXIS": "TRUNCATED_AXES",
    "TAMPERED_ASPECT_RATIO": "ASPECT_RATIO_DISTORTION",
    "DUAL_Y_AXES": "DUAL_AXES",
}
def _canon(code: str) -> str:
    return _CANON_MAP.get(code, code)

# Descriptions for summary-generation (kept short & neutral).
RULE_DESCRIPTIONS = {
    "MISSING_TITLE": "No descriptive chart title.",
    "MISSING_XLABEL": "X-axis lacks a clear label.",
    "MISSING_YLABEL": "Y-axis lacks a clear label.",
    "MISSING_LEGEND": "Multiple series without a legend.",
    "FONTSIZE_TOO_SMALL": "Text is likely too small to read.",
    "FIGSIZE_TOO_SMALL": "Figure size may hinder readability.",
    "INSUFFICIENT_COLOR_CONTRAST": "Colors may have poor contrast.",
    "ANIMATIONS": "Animated content may reduce accessibility.",
    "3D_EFFECTS": "3D effects may distort comparisons.",
    # Canonicalized names below:
    "INVERTED_AXES": "An axis is inverted, flipping increases/decreases.",
    "TRUNCATED_AXES": "Axis baseline is truncated, exaggerating differences.",
    "ASPECT_RATIO_DISTORTION": "Aspect ratio may distort slopes or trends.",
    "DUAL_AXES": "Dual y-axes can imply spurious relationships.",
    # Contextual-only extras:
    "BIASED_TITLE": "Title wording appears biased or emotive.",
    "MISLEADING_ANNOTATIONS": "Annotations imply unsupported relationships.",
    "DECEPTIVE_LABELS": "Axis labels are vague or confusing.",
    "FRAMING_BIAS": "External framing may bias interpretation.",
    "NON_SEQUENTIAL_AXIS": "Axis categories are out of logical order.",
}

# ---------- Helpers for summaries ----------
def _codes_to_lines(codes):
    # Turn ["A","B"] into ["A: desc", ...]
    out = []
    for c in codes:
        desc = RULE_DESCRIPTIONS.get(c, "")
        out.append(f"{c}: {desc}".strip())
    return out

_SUMMARY_PROMPT_TEMPLATE = """You are evaluating chart quality and accessibility.

You'll receive a list of rule violations (RULE_CODE: description). Write a concise, neutral summary (2–4 sentences) describing the potential issues these violations could cause for readers. Do not invent data, chart types, or context beyond the violations listed.

{extra}Violations:
{violations}

Return only the summary text (no bullets, no quotes).
"""

def _safe_gemini_call(prompt: str) -> str:
    if not HAVE_GEMINI:
        return ""
    # Simple throttle (single-threaded)
    global _LLM_LAST
    elapsed = time.time() - _LLM_LAST
    if elapsed < LLM_MIN_INTERVAL:
        time.sleep(max(0.0, LLM_MIN_INTERVAL - elapsed))
    try:
        res = client.generate_content(prompt)
        _LLM_LAST = time.time()
        text = (res.candidates[0].content.parts[0].text or "").strip()
        return text.replace("\n", " ").strip()
    except Exception as e:
        # Quiet failures → empty summary (keeps CSV clean)
        msg = str(e)
        # Mild backoff on typical transient errors
        if ("ResourceExhausted" in msg) or ("429" in msg) or ("quota" in msg.lower()) or ("rate" in msg.lower()) or ("timeout" in msg.lower()):
            time.sleep(LLM_MIN_INTERVAL * 1.5)
        return ""

def generate_summary_from_rules(codes, withhold=None) -> str:
    # Build visible list (omit withheld code)
    visible = [c for c in codes if c != withhold]
    if not visible:
        return ""
    violations_text = "\n".join(_codes_to_lines(visible))
    extra = ""
    if withhold:
        # Make it explicit that it must not hint at the withheld item
        extra = f"CRITICAL: Do NOT mention or allude to this violation: {withhold}: {RULE_DESCRIPTIONS.get(withhold, withhold)}.\n"
    prompt = _SUMMARY_PROMPT_TEMPLATE.format(violations=violations_text, extra=extra)
    return _safe_gemini_call(prompt)

def pick_masked_code(codes, row_id: int):
    """Deterministic 'random' picker so resume/streaming is stable."""
    if not codes:
        return None
    codes_sorted = sorted(set(codes))
    # Stable hash of row_id + codes to pick an index
    h = hashlib.md5( (str(row_id) + "|" + "|".join(codes_sorted)).encode("utf-8") ).hexdigest()
    idx = int(h[:8], 16) % len(codes_sorted)
    return codes_sorted[idx]

# ---------- Non-contextual rules ----------
def find_noncontextual_flaws(mpl_file: str):
    flaws = []
    mpl_index = mpl_file.find("import matplotlib")
    if mpl_index == -1:
        mpl_index = mpl_file.find("from matplotlib import")
    if mpl_index == -1:
        return flaws
    scan_text = mpl_file[mpl_index:]

    # Descriptive labels
    for fn in ["title", "xlabel", "ylabel"]:
        match = re.search(rf"{fn}\s*\(\s*['\"]([^'\"]*)['\"]", scan_text)
        if match:
            txt = match.group(1).strip().lower()
            if not txt or txt in ["x", "y", "series 1"]:
                flaws.append(f"MISSING_{fn.upper()}")
        else:
            flaws.append(f"MISSING_{fn.upper()}")

    # Legend required when multiple series
    plot_count = len(re.findall(r'plot\s*\(', scan_text))
    scatter_count = len(re.findall(r'scatter\s*\(', scan_text))
    if (plot_count + scatter_count > 1) and 'legend(' not in scan_text:
        flaws.append("MISSING_LEGEND")

    # Font size
    font_matches = re.findall(r'fontsize\s*=\s*(\d+)', scan_text)
    if any(int(size) < 15 for size in font_matches):
        flaws.append("FONTSIZE_TOO_SMALL")

    # Figure size
    fig_match = re.search(r'figsize\s*=\s*\(\s*([\d.]+)\s*,\s*([\d.]+)\s*\)', scan_text)
    if fig_match:
        w, h = float(fig_match.group(1)), float(fig_match.group(2))
        if w < 8 or h < 5:
            flaws.append("FIGSIZE_TOO_SMALL")

    # High-contrast colors
    color_matches = re.findall(r'color\s*=\s*[\'"]([^\'"]+)[\'"]', scan_text)
    safe_colors = {
        "#000000", "#0072B2", "#009E73", "#D55E00",
        "black", "blue", "green", "orange"
    }
    if any(color.lower() not in safe_colors for color in color_matches):
        flaws.append("INSUFFICIENT_COLOR_CONTRAST")

    # No animations
    if "FuncAnimation" in scan_text or "animation." in scan_text:
        flaws.append("ANIMATIONS")

    # Inverted Y-axis
    if re.search(r'\.\s*invert_yaxis\s*\(', scan_text):
        flaws.append("INVERTED_Y_AXIS")

    # Truncated Y-axis
    for m in re.finditer(r'(?:set_)?ylim\s*\(\s*([\-]?\d+(?:\.\d+)?)\s*,', scan_text):
        lower = float(m.group(1))
        if abs(lower) > 1e-6:
            flaws.append("TRUNCATED_Y_AXIS")
            break

    # 3D effects
    if (re.search(r'["\']\s*3d\s*["\']', scan_text) or
        "Axes3D" in scan_text or
        "plot_surface(" in scan_text):
        flaws.append("3D_EFFECTS")

    # Aspect ratio
    fig_aspect = re.search(r'figsize\s*=\s*\(\s*([\d.]+)\s*,\s*([\d.]+)\s*\)', scan_text)
    if fig_aspect:
        w, h = float(fig_aspect.group(1)), float(fig_aspect.group(2))
        if h != 0:
            ratio = w / h
            if ratio < 0.5 or ratio > 2.0:
                flaws.append("TAMPERED_ASPECT_RATIO")
    if re.search(r'set_aspect\s*\(|aspect\s*=', scan_text):
        flaws.append("TAMPERED_ASPECT_RATIO")

    # Dual Y
    if re.search(r'twin[xy]\s*\(', scan_text) or 'secondary_y=True' in scan_text:
        flaws.append("DUAL_Y_AXES")

    return list(sorted(set(flaws)))

# ---------- Contextual rules (LLM-inspected code) ----------
def generate_response(prompt: str) -> str:
    result = client.generate_content(prompt)
    candidate = result.candidates[0]
    return candidate.content.parts[0].text.strip()

_CONTEXTUAL_RULES_PROMPT_TEMPLATE = """You are an expert in data visualization integrity. I will provide you with:

1. A list of misleading visualization rules (each with a RULE_CODE and description),
2. A Matplotlib code snippet that generates a chart.

Your task:
- Analyze the code and detect which rules are violated based solely on what can be inferred from the code itself (e.g., axis behavior, titles, aspect ratio, annotations).
- Output only the list of violated RULE_CODEs in exactly this format: ["RULE_CODE1", "RULE_CODE2", ...]
- If the graph does not violate any rules, return: NONE

Rule Codes and Descriptions:

BIASED_TITLE:
A graph uses a biased or emotionally slanted title that influences interpretation before data is analyzed.

MISLEADING_ANNOTATIONS:
Annotations suggest causality or relationships that are not statistically or contextually justified.

DECEPTIVE_LABELS:
Y-axis or x-axis labels are vague, reversed, or omit key categories, leading to confusion.

FRAMING_BIAS:
External context or textual framing (e.g., comments, hashtags, plot subtitles) introduces bias not reflected in the graph.

INVERTED_AXES:
Y-axis is reversed (top to bottom), which misleads users by flipping the meaning of increases/decreases.

TRUNCATED_AXES:
Y-axis does not start at zero, which exaggerates visual differences.

ASPECT_RATIO_DISTORTION:
Aspect ratio is altered (e.g., too stretched or squished), making trends look steeper or flatter than they are.

DUAL_AXES:
Chart uses two different y-axes that may falsely suggest correlation between unrelated data series.

NON_SEQUENTIAL_AXIS:
X or Y axis uses a non-logical or out-of-order sequence (e.g., age ranges like 18-34, 45-55, 35-44).

Matplotlib Code:
{code}
"""

def find_contextual_flaws(mpl_file: str):
    # Bail if no matplotlib
    mpl_index = mpl_file.find("import matplotlib")
    if mpl_index == -1:
        mpl_index = mpl_file.find("from matplotlib import")
    if mpl_index == -1:
        return []
    scan_text = mpl_file[mpl_index:]

    if not HAVE_GEMINI:
        return []

    # Throttle one call at a time
    global _LLM_LAST
    elapsed = time.time() - _LLM_LAST
    if elapsed < LLM_MIN_INTERVAL:
        time.sleep(LLM_MIN_INTERVAL - elapsed)

    prompt = _CONTEXTUAL_RULES_PROMPT_TEMPLATE.format(code=scan_text)
    try:
        raw = generate_response(prompt)
        _LLM_LAST = time.time()
    except Exception as e:
        # Timeouts / 429 / transient network: skip this row so we keep moving
        msg = str(e)
        if ("ResourceExhausted" in msg) or ("429" in msg) or ("quota" in msg.lower()) or ("rate" in msg.lower()) or ("Timeout" in msg) or ("timed out" in msg.lower()):
            # small backoff
            time.sleep(LLM_MIN_INTERVAL * 1.5)
            return []
        return []

    if raw.strip().upper() == "NONE":
        return []

    try:
        parsed = ast.literal_eval(raw)
        if isinstance(parsed, list):
            return list(sorted(set(str(x).strip().upper() for x in parsed)))
    except Exception:
        # one more shot with stricter instruction
        try:
            raw2 = generate_response(prompt + '\nReturn ONLY a valid Python list literal of strings like ["RULE","RULE2"].')
            if raw2.strip().upper() == "NONE":
                return []
            parsed2 = ast.literal_eval(raw2)
            if isinstance(parsed2, list):
                return list(sorted(set(str(x).strip().upper() for x in parsed2)))
        except Exception:
            return []
    return []

# ---------- Rule columns ----------
NONCONTEXTUAL_CODES = [
    "MISSING_TITLE","MISSING_XLABEL","MISSING_YLABEL","MISSING_LEGEND",
    "FONTSIZE_TOO_SMALL","FIGSIZE_TOO_SMALL","INSUFFICIENT_COLOR_CONTRAST",
    "ANIMATIONS","INVERTED_Y_AXIS","TRUNCATED_Y_AXIS","3D_EFFECTS",
    "TAMPERED_ASPECT_RATIO","DUAL_Y_AXES",
]
CONTEXTUAL_CODES = [
    "BIASED_TITLE","MISLEADING_ANNOTATIONS","DECEPTIVE_LABELS","FRAMING_BIAS",
    "INVERTED_AXES","TRUNCATED_AXES","ASPECT_RATIO_DISTORTION","DUAL_AXES",
    "NON_SEQUENTIAL_AXIS",
]
ALL_RULE_COLS = NONCONTEXTUAL_CODES + CONTEXTUAL_CODES

# ---------- Load input ----------
fname = SRC if os.path.exists(SRC) else (SRC + '.csv' if os.path.exists(SRC + '.csv') else SRC)
try:
    df = pd.read_csv(fname)
except Exception:
    df = pd.read_csv(SRC + '.csv')
    fname = SRC + '.csv'

print(f"Loaded: {fname}  (rows={len(df)})")

# Detect code column
POSSIBLE_CODE_COLS = ["matplotlib_code","code","file_content","content","snippet","source","body","text"]
code_col = None
for c in POSSIBLE_CODE_COLS:
    if c in df.columns:
        code_col = c; break
if code_col is None:
    for c in df.columns:
        if df[c].dtype == object:
            sample = df[c].dropna().astype(str).head(10)
            if sample.str.contains("import matplotlib|from matplotlib", regex=True, case=False).any():
                code_col = c; break
if code_col is None:
    raise ValueError("Could not find a column with Matplotlib code. Add a column named e.g., 'matplotlib_code'.")

print(f"Using code column: {code_col}")

# ---- Random small sample (default 10) ----
if SAMPLE_N and len(df) > SAMPLE_N:
    df = df.sample(n=SAMPLE_N, random_state=42).reset_index(drop=True)
    print(f"Sampling: using {len(df)} rows (SAMPLE_N={SAMPLE_N})")

# ---------- Prepare streaming output ----------
base, ext = os.path.splitext(fname)
out = f"{base}_scored_stream_v2{ext or '.csv'}"  # new schema-safe name

# ---------- Small-sample runner (n=3 with progress bar) ----------
def run_small_sample(n: int = 3, seed: int = SAMPLE_SEED, out_override: str | None = None):
    """
    Randomly sample n rows from df, render full & masked images, generate two summaries,
    compute similarity (%), and append results to a CSV.
    Shows a progress bar if tqdm is installed.
    """
    # Choose output file (separate from your streaming file so it doesn't collide)
    base, ext = os.path.splitext(fname)
    dest = out_override or f"{base}_scored_sample{ext or '.csv'}"

    # Write header if needed
    if not os.path.exists(dest):
        pd.DataFrame(columns=header_cols).to_csv(dest, index=False)

    # Avoid reprocessing: read previously processed row_ids for the sample file
    processed_ids = set()
    try:
        prev = pd.read_csv(dest, usecols=["row_id"])
        processed_ids = set(prev["row_id"].dropna().astype(int).tolist())
    except Exception:
        pass

    # Take a random sample of n
    if n > len(df):
        n = len(df)
    sample_df = df.sample(n=n, random_state=seed)

    iterator = sample_df.iterrows()
    if tqdm:
        iterator = tqdm(iterator, total=len(sample_df), desc=f"Scoring {n} rows", unit="row")

    done = 0
    for idx, row in iterator:
        row_id = int(idx)
        if row_id in processed_ids:
            if tqdm: iterator.set_postfix_str(f"skip row_id={row_id} (already done)")
            continue

        code_raw = row[code_col] if isinstance(row[code_col], str) else ""
        mpl_index = code_raw.find("import matplotlib")
        if mpl_index == -1:
            mpl_index = code_raw.find("from matplotlib import")
        code = code_raw[mpl_index:] if mpl_index != -1 else code_raw

        # Detect flaws
        nonctx_list = find_noncontextual_flaws(code)
        ctx_list = find_contextual_flaws(code) if HAVE_GEMINI else []

        # Build base row
        out_row = dict(row)
        out_row["row_id"] = row_id
        for r in NONCONTEXTUAL_CODES:
            out_row[r] = 1 if r in nonctx_list else 0
        for r in CONTEXTUAL_CODES:
            out_row[r] = 1 if r in ctx_list else 0
        out_row["NONCONTEXTUAL_LIST"] = ";".join(nonctx_list)
        out_row["CONTEXTUAL_LIST"] = ";".join(ctx_list)

        # Render images
        full_img = masked_img = ""
        full_prefix = f"row{row_id}_full"
        imgs, _err = render_images_from_code(code, full_prefix, timeout=GRAPH_TIMEOUT)
        if imgs:
            full_img = imgs[0]

        # Pick rule to mask and patch if applicable
        detected_all = sorted(set(nonctx_list + ctx_list))
        masked_rule = _pick_masked_rule(row_id, detected_all)
        masked_prefix = f"row{row_id}_masked_{masked_rule or 'none'}"
        if masked_rule in NONCONTEXTUAL_CODES:
            patched = patch_code_for_mask(code, masked_rule)
            m_imgs, _merr = render_images_from_code(patched, masked_prefix, timeout=GRAPH_TIMEOUT)
            if m_imgs:
                masked_img = m_imgs[0]
        else:
            masked_img = full_img  # contextual/no rule → same image

        # Summaries
        full_summary = masked_summary = ""
        if full_img:
            full_summary = summarize_from_image(full_img, withhold_rule="") if HAVE_GEMINI else ""
        if masked_img:
            withhold = masked_rule if masked_rule in CONTEXTUAL_CODES else ""
            masked_summary = summarize_from_image(masked_img, withhold_rule=withhold) if HAVE_GEMINI else ""

        # Similarity (percentage 0..100)
        sim_val = _cosine_sim_pct(full_summary, masked_summary) if (full_summary and masked_summary) else ""

        # Fill new columns
        out_row["FULL_SUMMARY"] = full_summary
        out_row["MASKED_SUMMARY"] = masked_summary
        out_row["MASKED_RULE"] = masked_rule
        out_row["SUMMARY_SIMILARITY"] = sim_val
        out_row["SUMMARY_PAIR_ORDER"] = "sentence1=FULL, sentence2=MASKED"
        out_row["FULL_IMAGE"] = full_img
        out_row["MASKED_IMAGE"] = masked_img

        # Append to sample CSV
        pd.DataFrame([out_row])[header_cols].to_csv(dest, mode="a", header=False, index=False)

        done += 1
        if tqdm:
            postfix = f"id={row_id} rule={masked_rule or 'none'} sim={sim_val}"
            iterator.set_postfix_str(postfix)

    msg = f"✅ Small-sample run complete. Appended {done} rows → {dest}"
    print(msg)
    if tqdm:
        iterator.close()

# Build a unified header: row_id + original cols + rule flags + list cols + summary cols
orig_cols = list(df.columns)
summary_cols = ["MASKED_CODE","SUMMARY_ALL_RULES","SUMMARY_MASKED","ALL_CODES_LIST"]
header_cols = (["row_id"] + orig_cols +
               [c for c in ALL_RULE_COLS if c not in orig_cols] +
               ["NONCONTEXTUAL_LIST","CONTEXTUAL_LIST"] + summary_cols)

# Resume support: find already processed row_ids
processed_ids = set()
if os.path.exists(out):
    try:
        prev = pd.read_csv(out, usecols=["row_id"])
        processed_ids = set(prev["row_id"].dropna().astype(int).tolist())
        print(f"Resuming; already have {len(processed_ids)} rows in {out}")
    except Exception as e:
        print(f"Note: couldn't read existing {out} for resume ({e}); starting fresh.")

# Write header if new file
if not os.path.exists(out):
    pd.DataFrame(columns=header_cols).to_csv(out, index=False)

# ---------- Main loop (streaming) ----------
def score_row(code: str):
    if not isinstance(code, str):
        return [], []
    nonctx = find_noncontextual_flaws(code)
    ctx = find_contextual_flaws(code) if HAVE_GEMINI else []
    return nonctx, ctx

rows_done = 0
if __name__ == "__main__":
    # Small sample of 3 rows with a progress bar
    run_small_sample(n=3)


print(f"✅ Done. Appended {rows_done} new rows → {out}")


In [ ]:
df3 = pd.read_csv("github_matplotlib_audit_1_scored_stream_v2.csv")
df3.head()